<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 7 · Tablas dinámicas que responden preguntas
**Excel Corporativo · 12 Claves** — Resumir, comparar y graficar sin errores de agregación · ⏱ 40 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/07_tablas_dinamicas.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-07)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Armar una tabla dinámica eligiendo filas, columnas y valor.
2. Elegir el gráfico adecuado para cada pregunta de negocio.
3. Evitar los tres errores clásicos: anuladas, líneas contadas como facturas y promedio de porcentajes.
4. Comparar la venta con la meta y redactar el hallazgo.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · Tu propia tabla dinámica
Elige filas, columnas, valor y cómo mostrarlo, igual que en el panel de campos de Excel. Ejecuta la celda con ▶. Debajo de la tabla aparece el gráfico que mejor responde esa combinación y la razón de la elección.

In [ ]:
# @title Constructor de tabla dinámica
filas = "ciudad"  # @param ["ciudad", "categoria", "segmento", "canal", "nombre_vendedor", "mes", "anio"]
columnas = "anio"  # @param ["(ninguna)", "anio", "canal", "segmento", "categoria"]
valor = "Venta neta (suma de subtotal)"  # @param ["Venta neta (suma de subtotal)", "Margen bruto (suma)", "Margen %", "Líneas (recuento)", "Facturas (recuento distinto)"]
mostrar_como = "Valor"  # @param ["Valor", "% del total general", "% del total de columna", "% del total de fila"]
filtro_anio = "Todos"  # @param ["Todos", "2024", "2025"]
incluir_anuladas = False  # @param {type:"boolean"}

base = df if incluir_anuladas else validas
if filtro_anio != "Todos": base = base[base.anio == int(filtro_anio)]
col = None if columnas == "(ninguna)" or columnas == filas else columnas
def resumir(g):
    if valor.startswith("Venta"): return g["subtotal"].sum()
    if valor.startswith("Margen bruto"): return g["margen_bruto"].sum()
    if valor == "Margen %": return g["margen_bruto"].sum() / g["subtotal"].sum()
    if valor.startswith("Líneas"): return len(g)
    return g["id_factura"].nunique()
claves = [filas] + ([col] if col else [])
tabla = base.groupby(claves).apply(resumir)
tabla = tabla.unstack(col) if col else tabla.to_frame(valor.split(" (")[0])
if valor != "Margen %" and mostrar_como != "Valor":
    if mostrar_como == "% del total general": tabla = tabla / tabla.values.sum()
    elif mostrar_como == "% del total de columna": tabla = tabla / tabla.sum()
    else: tabla = tabla.div(tabla.sum(axis=1), axis=0)
es_pct = valor == "Margen %" or mostrar_como != "Valor"
if filas == "mes": tabla.index = [MESES[m - 1] for m in tabla.index]
elif filas != "anio": tabla = tabla.loc[tabla.sum(axis=1).sort_values(ascending=False).index] if col else tabla.sort_values(tabla.columns[0], ascending=False)
display(tabla.style.format(porc if es_pct else (lambda v: f"{v:,.0f}".replace(",", "."))))

# Elección del gráfico según la combinación
largo = tabla.reset_index().melt(id_vars=tabla.index.name or "index", var_name="serie", value_name="v")
x = tabla.index.name or "index"
if filas == "mes":
    fig = px.line(largo, x=x, y="v", color="serie" if col else None, markers=True); razon = "Las filas son meses: es tiempo, así que se usan líneas."
elif col and tabla.shape[1] > 3:
    grande = (not es_pct) and tabla.values.max() > 1e6
    fig = px.imshow(tabla / 1e6 if grande else tabla, text_auto=".0%" if es_pct else ".0f", aspect="auto", color_continuous_scale=["#EEF2FF", AZUL, "#00006A"])
    razon = "Hay más de tres columnas: un mapa de calor muestra el patrón sin saturar de colores." + (" Valores en millones de pesos." if grande else "")
elif col and mostrar_como == "% del total de fila":
    fig = px.bar(largo, y=x, x="v", color="serie", orientation="h", barmode="stack"); razon = "Porcentajes de cada fila que suman 100 %: barras apiladas al 100 %."
elif col:
    fig = px.bar(largo, x=x, y="v", color="serie", barmode="group", color_discrete_sequence=[GRIS, AZUL, TEAL]); razon = "Pocas series para comparar dentro de cada categoría: columnas agrupadas."
else:
    fig = px.bar(largo.sort_values("v"), y=x, x="v", orientation="h", color_discrete_sequence=[AZUL]); razon = "Una sola medida por categoría: barras horizontales ordenadas."
fig.update_layout(title=f"{valor.split(' (')[0]} por {filas}" + (f" y {col}" if col else ""), legend_title="", xaxis_title="", yaxis_title="", height=420)
if es_pct and not isinstance(fig.data[0], go.Heatmap): fig.update_layout(xaxis_tickformat=".0%" if not filas == "mes" else None, yaxis_tickformat=".0%" if filas == "mes" else None)
fig.show()
display(Markdown(f"**¿Por qué este gráfico?** {razon}"))
if incluir_anuladas: display(Markdown("⚠️ Estás incluyendo facturas anuladas: no son venta."))
if valor.startswith("Facturas") and col: display(Markdown("Ojo: los recuentos distintos de cada columna **no se suman** para obtener el total; una factura puede tener productos de varias categorías."))

## 2 · Seis preguntas de negocio, seis gráficos

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | P1 · ¿Qué ciudad vendió más en 2025? |
> | **Qué se grafica** | Barras: venta neta 2025 por ciudad, sin anuladas |
> | **Por qué esta forma** | Ranking de una medida en pocas categorías. |
> | **Qué se descartó** | Torta: seis partes parecidas no se comparan bien. |
> | **Cómo se lee** | La barra resaltada lidera. Compara las distancias entre barras. |
> | **En Excel 365** | Ciudad en Filas, subtotal en Valores, estado en Filtros, año 2025. |

In [ ]:
# @title P1 · Venta 2025 por ciudad
p1 = validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum().sort_values()
fig = go.Figure(go.Bar(x=p1.values / 1e6, y=p1.index, orientation="h", marker_color=[GRIS] * 5 + [AZUL],
                       text=[millones(v) for v in p1.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"{p1.index[-1]} vendió el {porc(p1.iloc[-1] / p1.sum())} de la venta neta 2025", xaxis_title=MM, height=360)
dar_aire(fig, p1.max() / 1e6)
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | P2 · ¿Cuánto creció cada ciudad de 2024 a 2025? |
> | **Qué se grafica** | Columnas agrupadas: 2024 (gris) y 2025 (azul) por ciudad, ordenadas por crecimiento |
> | **Por qué esta forma** | Dos series por categoría; el gris para el pasado deja el foco en el presente. |
> | **Qué se descartó** | Una línea por ciudad: dos puntos no hacen una tendencia. |
> | **Cómo se lee** | Lee la etiqueta de crecimiento sobre cada par de barras. |
> | **En Excel 365** | Ciudad en Filas, año en Columnas › Columna agrupada. |

In [ ]:
# @title P2 · Crecimiento por ciudad
p2 = validas.pivot_table(index="ciudad", columns="anio", values="subtotal", aggfunc="sum")
p2["var"] = p2[2025] / p2[2024] - 1
p2 = p2.sort_values("var", ascending=False)
fig = go.Figure([go.Bar(name="2024", x=p2.index, y=p2[2024] / 1e6, marker_color=GRIS), go.Bar(name="2025", x=p2.index, y=p2[2025] / 1e6, marker_color=AZUL)])
for c, f in p2.iterrows():
    fig.add_annotation(x=c, y=max(f[2024], f[2025]) / 1e6, text=porc(f["var"], signo=True), showarrow=False, yshift=12)
fig.update_layout(barmode="group", title=f"{p2.index[0]} fue la ciudad que más creció ({porc(p2['var'].iloc[0], signo=True)})", yaxis_title=MM, legend_title="")
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | P3 · ¿Qué línea deja más margen porcentual? |
> | **Qué se grafica** | Barras: margen % por línea, bien calculado (Σ margen ÷ Σ venta) y mal calculado (promedio de %) |
> | **Por qué esta forma** | Mostrar los dos cálculos juntos hace visible el error más común con porcentajes. |
> | **Qué se descartó** | Solo el valor correcto: el estudiante no vería por qué el otro está mal. |
> | **Cómo se lee** | Compara las dos barras de cada línea: la diferencia es el error de promediar porcentajes. |
> | **En Excel 365** | Campo calculado o margen_bruto ÷ subtotal fuera de la dinámica. |

In [ ]:
# @title P3 · Margen % bien y mal calculado
v = validas.assign(pct=validas.margen_bruto / validas.subtotal)
p3 = pd.DataFrame({"Correcto (Σ margen ÷ Σ venta)": v.groupby("categoria").margen_bruto.sum() / v.groupby("categoria").subtotal.sum(),
                   "Incorrecto (promedio de %)": v.groupby("categoria").pct.mean()}).sort_values("Correcto (Σ margen ÷ Σ venta)")
largo = p3.reset_index().melt(id_vars="categoria", var_name="cálculo", value_name="margen")
fig = px.bar(largo, y="categoria", x="margen", color="cálculo", barmode="group", orientation="h", color_discrete_sequence=[AZUL, GRIS])
fig.update_layout(title=f"{p3.index[-1]} deja el mayor margen; {p3.index[0]}, el menor ({porc(p3.iloc[0, 0])})", xaxis_tickformat=".0%", yaxis_title="", xaxis_title="", legend_title="", height=400)
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | P4 · ¿El e-commerce ganó participación? |
> | **Qué se grafica** | Barras apiladas al 100 %: reparto de la venta por canal en cada año |
> | **Por qué esta forma** | Composición de un total en pocas partes, comparada entre dos años. |
> | **Qué se descartó** | Valores absolutos: ocultan que el total creció. |
> | **Cómo se lee** | Compara el ancho del segmento magenta en 2024 y en 2025. |
> | **En Excel 365** | Año en Filas, canal en Columnas, Mostrar valores como › % del total de fila › Barra 100 % apilada. |

In [ ]:
# @title P4 · Participación por canal
p4 = pd.crosstab(validas.anio, validas.canal, values=validas.subtotal, aggfunc="sum", normalize="index")[["Presencial", "Televentas", "E-commerce"]]
largo = p4.reset_index().melt(id_vars="anio", var_name="canal", value_name="part"); largo["anio"] = largo["anio"].astype(str)
fig = px.bar(largo, y="anio", x="part", color="canal", orientation="h", text=largo["part"].map(lambda v: f"{v:.0%}"), color_discrete_sequence=[AZUL, TEAL, MAGENTA])
fig.update_traces(marker_line_color="white", marker_line_width=2, insidetextanchor="middle")
fig.update_layout(barmode="stack", title=f"El e-commerce pasó de {porc(p4.loc[2024, 'E-commerce'])} a {porc(p4.loc[2025, 'E-commerce'])} de la venta",
                  xaxis_tickformat=".0%", yaxis_title="", xaxis_title="", legend_title="", height=300, yaxis=dict(autorange="reversed"))
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | P5 · ¿Cuántas facturas emitió cada sede? |
> | **Qué se grafica** | Columnas agrupadas: líneas (recuento) y facturas (recuento distinto) por sede |
> | **Por qué esta forma** | Muestra en el mismo gráfico la diferencia entre contar filas y contar facturas. |
> | **Qué se descartó** | Solo facturas: no se vería el error de contar filas. |
> | **Cómo se lee** | La barra de líneas siempre es más alta: cada factura tiene varios productos. |
> | **En Excel 365** | Recuento distinto requiere marcar «Agregar al Modelo de datos». |

In [ ]:
# @title P5 · Líneas frente a facturas
p5 = validas.groupby("ciudad").agg(lineas=("id_linea", "size"), facturas=("id_factura", "nunique")).sort_values("facturas", ascending=False)
fig = go.Figure([go.Bar(name="Líneas (recuento)", x=p5.index, y=p5.lineas, marker_color=GRIS), go.Bar(name="Facturas (recuento distinto)", x=p5.index, y=p5.facturas, marker_color=AZUL)])
fig.update_layout(barmode="group", title=f"Contar filas casi duplica las facturas: {p5.lineas.sum() / p5.facturas.sum():.1f} líneas por factura".replace(".", ","), yaxis_title="", legend_title="")
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | P6 · ¿En qué meses y sedes se concentra la venta? |
> | **Qué se grafica** | Mapa de calor: sede × mes de 2025, color más oscuro = más venta |
> | **Por qué esta forma** | Dos categorías cruzadas: el color revela patrones que una tabla de 72 números esconde. |
> | **Qué se descartó** | Doce líneas, una por mes: imposible de leer. |
> | **Cómo se lee** | Busca columnas oscuras (meses fuertes) y filas claras (sedes pequeñas). |
> | **En Excel 365** | Tabla dinámica sede × mes › Formato condicional › Escalas de color. |

In [ ]:
# @title P6 · Mapa de calor sede × mes
p6 = validas[validas.anio == 2025].pivot_table(index="ciudad", columns="mes", values="subtotal", aggfunc="sum").loc[ORDEN_CIUDADES] / 1e6
i, j = np.unravel_index(np.argmax(p6.values), p6.shape)
fig = px.imshow(p6, x=MESES, y=p6.index, text_auto=".0f", aspect="auto", color_continuous_scale=["#EEF2FF", "#9AA8F0", AZUL, "#00006A"])
fig.update_layout(title=f"{p6.index[i]} en {MESES[j].lower()} es el punto más alto de 2025 (millones de pesos)", coloraxis_colorbar_title="M$", xaxis_title="", yaxis_title="", height=380)
fig.show()

## 3 · Real contra meta

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué sedes cumplieron la meta de 2025? |
> | **Qué se grafica** | Barras: % de cumplimiento por sede · Línea punteada: 100 % · Color: cumplió (teal) o no (magenta) |
> | **Por qué esta forma** | El cumplimiento es una razón; ordenar por ella responde directamente la pregunta. |
> | **Qué se descartó** | Barras de real y meta en pesos: Bogotá dominaría el gráfico aunque no sea la que más cumplió. |
> | **Cómo se lee** | Las barras que pasan la línea punteada cumplieron. El texto da el porcentaje exacto. |
> | **En Excel 365** | SUMAR.SI.CONJUNTO de real y meta por sede, y luego Barra con línea de referencia. |

In [ ]:
# @title Cumplimiento 2025 por sede
real = validas[validas.anio == 2025].groupby("id_sede")["subtotal"].sum()
meta = metas[metas.anio == 2025].groupby("id_sede")["meta_subtotal"].sum()
cum = (real / meta).rename(index=sedes.set_index("id_sede")["ciudad"]).sort_values()
fig = go.Figure(go.Bar(x=cum.values, y=cum.index, orientation="h", marker_color=[TEAL if v >= 1 else MAGENTA for v in cum.values],
                       text=[("✓ " if v >= 1 else "✗ ") + porc(v) for v in cum.values], textposition="outside", cliponaxis=False))
fig.add_vline(x=1, line_dash="dot", line_color="#11153A", annotation_text="Meta 100 %")
fig.update_layout(title=f"{(cum >= 1).sum()} de 6 sedes cumplieron; {cum.index[0]} quedó en {porc(cum.iloc[0])}", xaxis_tickformat=".0%",
                  xaxis_range=[0.8, max(1.15, cum.max() + 0.06)], height=360)
fig.show()
mejor = cum.index[-1]

## 4 · Del resultado al informe
Una tabla dinámica no es un informe. El informe son tres frases con cifra, comparación y siguiente pregunta:

In [ ]:
# @title Generar el borrador del hallazgo
display(Markdown(f"""
1. En 2025 la empresa vendió **{millones(p2[2025].sum())}** netos, **{porc(p2[2025].sum() / p2[2024].sum() - 1, signo=True)}** frente a 2024.
2. **{cum.index[-1]}** superó su meta ({porc(cum.iloc[-1])}) y **{cum.index[0]}** quedó en {porc(cum.iloc[0])}. ¿Qué explica la brecha de {cum.index[0]}?
3. El e-commerce ya es el **{porc(p4.loc[2025, 'E-commerce'])}** de la venta. ¿Tiene el mismo margen que los otros canales?
"""))

## 5 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Qué participación tuvo el e-commerce en la venta neta de 2025?
respuesta = "Elige una opción"  # @param ["Elige una opción", "13 %", "19 %", "24 %", "30 %"]
revisar(respuesta, f"{p4.loc[2025, 'E-commerce']:.0%}".replace("%", " %"), """Es el segmento magenta de la barra 2025 en P4.""")

In [ ]:
# @title Reto 2 · ¿Qué sede tuvo el mayor cumplimiento de meta en 2025?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
revisar(respuesta, mejor, """Es la barra más larga del gráfico de cumplimiento.""")

In [ ]:
# @title Reto 3 · En una tabla dinámica, ¿qué usas para contar facturas y no líneas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Recuento", "Suma", "Recuento distinto", "Promedio"]
revisar(respuesta, "Recuento distinto", """Recuento cuenta filas; cada factura ocupa varias filas.""")

## Apóyate en IA
Usa la IA para pasar de la pregunta a la configuración de la tabla dinámica:

```text
Tengo en Excel 365 una Tabla tVentas con: fecha, id_factura, ciudad, canal, categoria, subtotal, margen_bruto, estado.
Pregunta de negocio: "[escribe tu pregunta]".
Dime qué va en Filas, Columnas, Valores y Filtros, qué "Mostrar valores como" usar, qué gráfico elegir y por qué,
y qué error de agregación debo evitar. No inventes resultados.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Primero la pregunta, después la tabla dinámica.
* Sin anuladas; facturas con recuento distinto; porcentajes como razón de sumas.
* El gráfico se elige por la pregunta: ranking, tiempo, composición o cumplimiento.

**Siguiente:** [Laboratorio Excel + Colab](https://omarli-360.github.io/excel-corporativo-12-claves/#laboratorio)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>